# CVaR / Expected Shortfall: Sizing the Losses That Live Beyond VaR

Value-at-Risk tells you where the tail begins. It says nothing about what lives inside it — and inside the tail is where portfolios actually die. Two books can report identical 99% VaR yet differ threefold in what they lose once the threshold breaks; for fat-tailed assets like high-yield credit, that difference is the whole risk. Expected shortfall (CVaR) fixes the blind spot by averaging the tail instead of pointing at its door, which is why Basel made it the regulatory standard. We estimate both, historically and with a Student-t, on 4,462 days of HYG and VWO — a sample that deliberately includes the crisis these measures were built for.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

plt.rcParams["figure.figsize"] = (10, 5)

## 1. What VaR cannot see

VaR at confidence α is a **quantile**: the smallest loss exceeded on only the worst (1−α) of days. Read that definition again and notice what it does not say — nothing about **how much** you lose on those days. VaR is a threshold, not an average, so it is structurally blind to everything beyond itself. A ten-line thought experiment makes the blindness concrete: build two 1,000-day P&L histories that agree on 990 benign days **and** on the day that sets the 99% quantile, but whose ten worst days differ by a factor of six.

Both books report a 99% VaR of 2.40% , so a risk report built on VaR alone calls them equally risky. Yet book A's CVaR is 2.60% against book B's 7.26% , with worst days of −2.60% and −15.00% — identical VaR, 2.8 × the expected tail loss. CVaR — the **mean** loss conditional on breaching the VaR quantile — separates the two books instantly, because it averages over the whole tail instead of reading one point at its edge.

In [ ]:
px = yf.download(["HYG", "VWO"], start="2007-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].dropna()
rets = px.pct_change().dropna()
rets["PORT"] = 0.5 * rets["HYG"] + 0.5 * rets["VWO"]   # daily-rebalanced 50/50
rets.describe().T[["mean", "std", "min", "max"]]

In [ ]:
def var_hist(x, a):
    return -np.quantile(x, 1 - a)

def cvar_hist(x, a):
    q = np.quantile(x, 1 - a)
    return -x[x <= q].mean()

body   = np.linspace(-0.024, 0.024, 990)          # identical benign body
tail_a = np.full(10, -0.026)                      # thin tail
tail_b = np.array([-0.026, -0.03, -0.04, -0.05, -0.06,
                   -0.07, -0.08, -0.10, -0.12, -0.15])   # fat tail
da, db = np.r_[body, tail_a], np.r_[body, tail_b]

for name, d in [("thin tail", da), ("fat tail", db)]:
    print(f"{name}: 99% VaR = {var_hist(d, .99):.2%}   99% CVaR = {cvar_hist(d, .99):.2%}")

## 2. Two fat-tailed assets

Now the real thing. HYG (iShares high-yield corporate bond ETF) and VWO (Vanguard emerging-markets equity) are chosen because they are risky in opposite ways. EM equity is honestly volatile — it looks risky and it is. Credit is the trap: it collects small, steady coupons in calm markets and hands them back all at once in a crisis. On paper HYG's 11.2% annualised vol looks far safer than VWO's 27.1% — until you notice that the quiet series hides a worst day of -8.10% . The aligned sample runs 2007-04-11 (HYG's listing) to 2024-12-31 . A Student-t fitted to HYG's daily returns lands at ν ≈ 1.98 degrees of freedom — below 2, which means the fitted distribution does not even possess a finite variance. Read that as a diagnostic as much as an estimate: an unconditional iid fit has nowhere to put 2008's volatility clustering except the tail parameter, so it buys realism at the extremes by overstating how wild a **typical** day is. (Condition on a GARCH filter and the residual df comes out higher; the unconditional fit is the honest worst case.)

The horizontal gap between the two dashed lines — 2.09% to 3.37% — is everything VaR does not price, and HYG's gap is the widest of the three books we measure.

## 3. Historical and Student-t estimates

Estimating CVaR comes down to two philosophies. The historical estimator trusts the data: sort, take the quantile, average everything beyond it. The parametric route trusts the model: fit a Student-t and use **Acerbi's closed-form expected shortfall** — the analytic mean of the t's tail — which can extrapolate to days worse than anything the sample has actually seen:

Three things to read off that table:

- **The FRTB calibration at work** — HYG's 97.5% CVaR ( 2.29% ) sits close to its 99% VaR ( 2.09% ): similar magnitude, but the CVaR number keeps growing when the tail does.
- **The t extrapolates past the sample** — its CVaR ( 4.09% ) exceeds the historical one because with ν ≈ 1.98 the fitted tail expects days worse than any yet observed.
- **The small-sample caveat** — at 99% the historical CVaR is the mean of just 45 observations. The estimator with the best theoretical properties stands on the fewest data points, which is the practical argument for fitting a parametric tail and letting it extrapolate.

Read the first and fifth columns together. HYG's volatility is well under half of VWO's, yet its CVaR/VaR ratio of 1.62 × is the **highest** in the table — credit's deceptively quiet variance hides the most disproportionate tail. Any tool that ranks these assets by σ alone has the risk ordering half wrong.

In [ ]:
def var_t(params, a):
    nu, loc, scale = params
    return -(loc + scale * stats.t.ppf(1 - a, nu))

def cvar_t(params, a):                      # Acerbi & Tasche closed form
    nu, loc, scale = params
    p = 1 - a
    xp = stats.t.ppf(p, nu)
    return -loc + scale * stats.t.pdf(xp, nu) * (nu + xp**2) / ((nu - 1) * p)

rows = []
for c in ["HYG", "VWO", "PORT"]:
    x = rets[c].values
    fit = stats.t.fit(x)
    rows.append({
        "asset": c, "nu": fit[0],
        "hist VaR99": var_hist(x, .99),  "hist CVaR99": cvar_hist(x, .99),
        "t VaR99":    var_t(fit, .99),   "t CVaR99":    cvar_t(fit, .99),
        "hist CVaR97.5": cvar_hist(x, .975),
        "CVaR/VaR": cvar_hist(x, .99) / var_hist(x, .99),
        "worst day": x.min(),
    })
pd.DataFrame(rows).set_index("asset").round(4)

In [ ]:
x = rets["HYG"].values
nu, loc, scale = stats.t.fit(x)
edges = np.linspace(np.quantile(x, 0.0005), np.quantile(x, 0.9995), 57)
plt.hist(x, bins=edges, density=True, alpha=0.45, label="HYG daily returns")
grid = np.linspace(edges[0], edges[-1], 400)
plt.plot(grid, stats.t.pdf(grid, nu, loc, scale), lw=2, label=f"Student-t (nu={nu:.1f})")
plt.axvline(-var_hist(x, .99),  ls="--", c="firebrick", label="99% VaR")
plt.axvline(-cvar_hist(x, .99), ls="--", c="darkorange", label="99% CVaR")
plt.legend(); plt.title("HYG daily returns — the gap between VaR and CVaR is the tail");
plt.show()

## 4. Subadditivity — the coherence test

In 1999, Artzner, Delbaen, Eber and Heath wrote down four axioms any sane risk measure should satisfy. VaR fails one — and it is the one your intuition cares most about: **subadditivity**, ρ(A+B) ≤ ρ(A) + ρ(B), the rule that diversification must never create risk. The classic counterexample needs only two independent bonds, each defaulting with probability 0.7%. Held alone, each has **zero** 99% VaR: a 0.7% loss probability hides entirely below the 1% threshold, so VaR literally cannot see it. Mix them 50/50 and the book loses on 1.4% of scenarios — above 1% — so the diversified book has strictly **positive**99% VaR. Diversifying “created” risk, says VaR. The risk was there all along, of course. VaR was hiding it.

It is not just a parlour trick. On the full HYG/VWO sample the 50/50 portfolio behaves — 99% VaR of 3.26% against a weighted blend of 3.47% . But scan calendar-year subsamples across confidence levels and historical VaR breaks subadditivity 64 times; the worst case is 2009 at α = 98.8% , where the portfolio's VaR of 4.48% exceeds the blend's 3.95% . CVaR, by construction — Rockafellar & Uryasev's convex formulation makes this explicit — never violates: 4.92% for the portfolio versus 5.26% for the blend at 99%, and it passes at every year and level where VaR fails.

In [ ]:
# empirical check on HYG/VWO at 99% (full sample)
for name, fn in [("VaR", var_hist), ("CVaR", cvar_hist)]:
    port  = fn(rets["PORT"].values, .99)
    blend = 0.5 * fn(rets["HYG"].values, .99) + 0.5 * fn(rets["VWO"].values, .99)
    print(f"99% {name}: portfolio = {port:.2%}   0.5·HYG + 0.5·VWO = {blend:.2%}"
          f"   subadditive: {port <= blend}")

# historical VaR violates subadditivity in calendar-year subsamples
hits = []
for y in range(2008, 2025):
    w = rets.loc[str(y)]
    for a in np.arange(0.90, 0.9951, 0.0025):
        vp = var_hist(w["PORT"].values, a)
        vb = 0.5 * var_hist(w["HYG"].values, a) + 0.5 * var_hist(w["VWO"].values, a)
        if vp > vb:
            hits.append((y, round(a, 4), vp, vb))
print(f"\nyear-level VaR subadditivity violations: {len(hits)}")
for y, a, vp, vb in sorted(hits, key=lambda t: t[3] - t[2])[:3]:
    print(f"  {y} @ alpha={a}: VaR(port) = {vp:.2%} > blend = {vb:.2%}")

# CVaR never violates — try it
assert not [1 for y, a, _, _ in hits
            if cvar_hist(rets.loc[str(y), "PORT"].values, a) >
               0.5 * cvar_hist(rets.loc[str(y), "HYG"].values, a)
             + 0.5 * cvar_hist(rets.loc[str(y), "VWO"].values, a)]

## 5. The GFC, seen from the tail

HYG's full-sample 99% VaR is 2.09% . Now watch the crisis ignore it: from September 2008 through March 2009, HYG breached that threshold on 24 of 146 trading days — a 1% tail arriving at 24 × its expected frequency of ~ 1.5 days. The worst prints came fast: -8.10% on 2008-09-29 , -6.67% on 2008-10-10 , -6.01% on 2008-09-17 — every one of them multiples of the VaR line, in an ETF marketed as a bond fund.

At the trough a dollar invested at HYG's listing was worth $ 0.69 . VaR answered "how often?" — and even that answer failed under regime change. CVaR at least asks the question that mattered in 2008: **how bad is it when it happens?**

In [ ]:
gwin = rets.loc["2007":"2010", "HYG"]
(1 + gwin).cumprod().plot(title="HYG cumulative return through the GFC")
for d, ret in gwin.nsmallest(5).items():
    print(f"{d.date()}  {ret: .2%}")

crisis = rets.loc["2008-09":"2009-03", "HYG"]
var99 = var_hist(rets["HYG"].values, .99)
print(f"\ndays beyond full-sample 99% VaR in Sep-08..Mar-09: "
      f"{(crisis < -var99).sum()} of {len(crisis)} (expected ~{0.01*len(crisis):.1f})")

## 6. Spending a tail budget: CVaR vs MV

So far CVaR has been the better thermometer. The sharper question is whether it changes what you **do**. With two assets, pure risk-minimising is degenerate here — volatility and tail agree that HYG is the quieter asset, so both `rm="MV"` and `rm="CVaR"` corner at 100% HYG. The measures diverge the moment you **spend a risk budget**. Hand two desks the same mandate — an expected loss on the worst 1% of days of at most 4% — and let each maximise return against it. The MV desk speaks only volatility, so it must translate the budget through a normal distribution (ES₉₉ = 2.665σ for a Gaussian, so σ ≤ 1.50% daily); the CVaR desk constrains the realised tail directly:

Same stated risk appetite, radically different books: 83.8% VWO through the variance lens versus 24.7% through the CVaR lens. Because both assets' tails are fatter than the Gaussian used in the translation, the MV desk's realised 99% CVaR of 6.37% overshoots its own 4% mandate by roughly 59 % — the tail it was told to cap is exactly what its risk measure could not see.

In [ ]:
import riskfolio as rp
from scipy import stats as st

BUDGET = 0.04                                    # 99% ES budget, daily
sig_budget = BUDGET / (st.norm.pdf(st.norm.ppf(0.01)) / 0.01)   # = BUDGET / 2.665
Y = rets[["HYG", "VWO"]]

p_mv = rp.Portfolio(returns=Y)
p_mv.assets_stats(method_mu="hist", method_cov="hist")
p_mv.upperdev = sig_budget                       # normal-translated vol cap
w_mv = p_mv.optimization(model="Classic", rm="MV", obj="MaxRet", hist=True)

p_cv = rp.Portfolio(returns=Y)
p_cv.assets_stats(method_mu="hist", method_cov="hist")
p_cv.alpha = 0.01
p_cv.upperCVaR = BUDGET                          # the tail budget itself
w_cv = p_cv.optimization(model="Classic", rm="CVaR", obj="MaxRet", hist=True)

for name, w in [("MV desk (vol-translated budget)", w_mv), ("CVaR desk", w_cv)]:
    pr = (Y * w["weights"].values).sum(axis=1).values
    print(f"{name}: HYG {w.loc['HYG', 'weights']:.1%} / VWO {w.loc['VWO', 'weights']:.1%}"
          f" -> realized 99% CVaR {cvar_hist(pr, .99):.2%} vs budget {BUDGET:.0%}")

- Artzner, P., Delbaen, F., Eber, J.-M. & Heath, D. (1999). Coherent Measures of Risk. Mathematical Finance 9(3), 203–228.
- Acerbi, C. & Tasche, D. (2002). On the coherence of expected shortfall. Journal of Banking & Finance 26(7), 1487–1503.
- Rockafellar, R.T. & Uryasev, S. (2000). Optimization of Conditional Value-at-Risk. Journal of Risk 2(3), 21–41.
- Acerbi, C. & Székely, B. (2014). Back-testing Expected Shortfall. Risk Magazine, December 2014.
- Basel Committee on Banking Supervision (2019). Minimum capital requirements for market risk (FRTB). Bank for International Settlements.
- Companion notebook: `cvar-expected-shortfall.ipynb` — reproduces every figure from raw data (fully deterministic; no simulation).